# Flight Price Prediction - Data Preprocessing and Feature Engineering

## Project Objective

The objective of this notebook is to prepare airline flight data for machine learning by performing data cleaning, preprocessing, feature engineering, and categorical encoding.

The dataset contains information such as airline, source city, destination, departure time, arrival time, duration, number of stops, and ticket price. Since machine learning models require numerical input features, several transformations are applied to convert raw data into a structured format suitable for predictive modeling.

This notebook focuses entirely on data preparation. Model training can be performed in a separate notebook using the processed dataset.

In [54]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
%matplotlib inline

## Load Training and Test Datasets

The project uses separate training and test datasets.

- The training dataset contains both predictor variables and the target variable (Price).
- The test dataset contains only predictor variables and is used for generating predictions after model development.

Both datasets are loaded and later combined so that identical preprocessing and feature engineering steps can be applied consistently.

In [55]:
train_df=pd.read_excel('Data_Train.xlsx')
train_df.head()

,Airline,Date_of_Journey,Source,Destination,Route,Dep_Time,Arrival_Time,Duration,Total_Stops,Additional_Info,Price
0,IndiGo,24/03/2019,Banglore,New Delhi,BLR → DEL,22:20,01:10 22 Mar,2h 50m,non-stop,No info,3897
1,Air India,1/05/2019,Kolkata,Banglore,CCU → IXR → BBI → BLR,05:50,13:15,7h 25m,2 stops,No info,7662
2,Jet Airways,9/06/2019,Delhi,Cochin,DEL → LKO → BOM → COK,09:25,04:25 10 Jun,19h,2 stops,No info,13882
3,IndiGo,12/05/2019,Kolkata,Banglore,CCU → NAG → BLR,18:05,23:30,5h 25m,1 stop,No info,6218
4,IndiGo,01/03/2019,Banglore,New Delhi,BLR → NAG → DEL,16:50,21:35,4h 45m,1 stop,No info,13302


In [56]:
train_df.shape

(10683, 11)

In [57]:
test_df=pd.read_excel('Test_set.xlsx')
test_df.head()

,Airline,Date_of_Journey,Source,Destination,Route,Dep_Time,Arrival_Time,Duration,Total_Stops,Additional_Info
0,Jet Airways,6/06/2019,Delhi,Cochin,DEL → BOM → COK,17:30,04:25 07 Jun,10h 55m,1 stop,No info
1,IndiGo,12/05/2019,Kolkata,Banglore,CCU → MAA → BLR,06:20,10:20,4h,1 stop,No info
2,Jet Airways,21/05/2019,Delhi,Cochin,DEL → BOM → COK,19:15,19:00 22 May,23h 45m,1 stop,In-flight meal not included
3,Multiple carriers,21/05/2019,Delhi,Cochin,DEL → BOM → COK,08:00,21:00,13h,1 stop,No info
4,Air Asia,24/06/2019,Banglore,Delhi,BLR → DEL,23:55,02:45 25 Jun,2h 50m,non-stop,No info


In [58]:
test_df.shape

(2671, 10)

## Combine Train and Test Data

The training and test datasets are concatenated into a single dataframe before preprocessing.

Combining the datasets ensures that all feature engineering, missing value handling, and categorical encoding are performed consistently across both datasets, avoiding differences between training and prediction data.

In [59]:
final_df=pd.concat([train_df,test_df], axis=0)
final_df.shape

(13354, 11)

In [60]:
final_df.columns

Index(['Airline', 'Date_of_Journey', 'Source', 'Destination', 'Route',
       'Dep_Time', 'Arrival_Time', 'Duration', 'Total_Stops',
       'Additional_Info', 'Price'],
      dtype='object')

In [61]:
final_df.dtypes

Airline             object
Date_of_Journey     object
Source              object
Destination         object
Route               object
Dep_Time            object
Arrival_Time        object
Duration            object
Total_Stops         object
Additional_Info     object
Price              float64
dtype: object

## Initial Data Exploration

Before preprocessing, the dataset structure is examined to understand:

- Number of observations
- Available features
- Data types
- Missing values
- Overall data quality

This helps identify which columns require cleaning or transformation.

In [34]:
final_df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 13354 entries, 0 to 2670
Data columns (total 11 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   Airline          13354 non-null  object 
 1   Date_of_Journey  13354 non-null  object 
 2   Source           13354 non-null  object 
 3   Destination      13354 non-null  object 
 4   Route            13353 non-null  object 
 5   Dep_Time         13354 non-null  object 
 6   Arrival_Time     13354 non-null  object 
 7   Duration         13354 non-null  object 
 8   Total_Stops      13353 non-null  object 
 9   Additional_Info  13354 non-null  object 
 10  Price            10683 non-null  float64
dtypes: float64(1), object(10)
memory usage: 1.2+ MB


In [35]:
final_df.describe()

,Price
count,10683.000000
mean,9087.064121
std,4611.359167
min,1759.000000
25%,5277.000000
50%,8372.000000
75%,12373.000000
max,79512.000000


In [62]:
final_df.tail()

,Airline,Date_of_Journey,Source,Destination,Route,Dep_Time,Arrival_Time,Duration,Total_Stops,Additional_Info,Price
2666,Air India,6/06/2019,Kolkata,Banglore,CCU → DEL → BLR,20:30,20:25 07 Jun,23h 55m,1 stop,No info,NaN
2667,IndiGo,27/03/2019,Kolkata,Banglore,CCU → BLR,14:20,16:55,2h 35m,non-stop,No info,NaN
2668,Jet Airways,6/03/2019,Delhi,Cochin,DEL → BOM → COK,21:50,04:25 07 Mar,6h 35m,1 stop,No info,NaN
2669,Air India,6/03/2019,Delhi,Cochin,DEL → BOM → COK,04:00,19:15,15h 15m,1 stop,No info,NaN
2670,Multiple carriers,15/06/2019,Delhi,Cochin,DEL → BOM → COK,04:55,19:15,14h 20m,1 stop,No info,NaN


## Feature Engineering from Journey Date

The original journey date contains day, month, and year together.

Machine learning algorithms generally perform better when temporal information is represented as separate numerical features. Therefore, the journey date is decomposed into:

- Day
- Month
- Year

These derived features allow the model to capture seasonal and monthly travel patterns.

In [65]:
final_df['Date']=final_df['Date_of_Journey'].apply(lambda x: x.split('/')[0])
final_df['Month']=final_df['Date_of_Journey'].apply(lambda x: x.split('/')[1])
final_df['Year']=final_df['Date_of_Journey'].apply(lambda x: x.split('/')[2])
final_df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 13354 entries, 0 to 2670
Data columns (total 14 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   Airline          13354 non-null  object 
 1   Date_of_Journey  13354 non-null  object 
 2   Source           13354 non-null  object 
 3   Destination      13354 non-null  object 
 4   Route            13353 non-null  object 
 5   Dep_Time         13354 non-null  object 
 6   Arrival_Time     13354 non-null  object 
 7   Duration         13354 non-null  object 
 8   Total_Stops      13353 non-null  object 
 9   Additional_Info  13354 non-null  object 
 10  Price            10683 non-null  float64
 11  Date             13354 non-null  object 
 12  Month            13354 non-null  object 
 13  Year             13354 non-null  object 
dtypes: float64(1), object(13)
memory usage: 1.5+ MB


In [66]:
#Converting Date, Month, Year into integer datatype using astype

In [67]:
final_df['Date']=final_df['Date'].astype(int)
final_df['Month']=final_df['Month'].astype(int)
final_df['Year']=final_df['Year'].astype(int)
final_df.dtypes

Airline             object
Date_of_Journey     object
Source              object
Destination         object
Route               object
Dep_Time            object
Arrival_Time        object
Duration            object
Total_Stops         object
Additional_Info     object
Price              float64
Date                 int64
Month                int64
Year                 int64
dtype: object

In [68]:
final_df.head(5)

,Airline,Date_of_Journey,Source,Destination,Route,Dep_Time,Arrival_Time,Duration,Total_Stops,Additional_Info,Price,Date,Month,Year
0,IndiGo,24/03/2019,Banglore,New Delhi,BLR → DEL,22:20,01:10 22 Mar,2h 50m,non-stop,No info,3897.0,24,3,2019
1,Air India,1/05/2019,Kolkata,Banglore,CCU → IXR → BBI → BLR,05:50,13:15,7h 25m,2 stops,No info,7662.0,1,5,2019
2,Jet Airways,9/06/2019,Delhi,Cochin,DEL → LKO → BOM → COK,09:25,04:25 10 Jun,19h,2 stops,No info,13882.0,9,6,2019
3,IndiGo,12/05/2019,Kolkata,Banglore,CCU → NAG → BLR,18:05,23:30,5h 25m,1 stop,No info,6218.0,12,5,2019
4,IndiGo,01/03/2019,Banglore,New Delhi,BLR → NAG → DEL,16:50,21:35,4h 45m,1 stop,No info,13302.0,1,3,2019


## Extract Arrival Time Components

Arrival time contains both the arrival time and arrival date.

Only the time component is required for modeling, so the arrival hour and minute are extracted separately and converted into numerical features.

In [71]:
final_df['Arrival_Time']=final_df['Arrival_Time'].str.split(' ').str[0]

In [72]:
final_df.head()

,Airline,Date_of_Journey,Source,Destination,Route,Dep_Time,Arrival_Time,Duration,Total_Stops,Additional_Info,Price,Date,Month,Year
0,IndiGo,24/03/2019,Banglore,New Delhi,BLR → DEL,22:20,01:10,2h 50m,non-stop,No info,3897.0,24,3,2019
1,Air India,1/05/2019,Kolkata,Banglore,CCU → IXR → BBI → BLR,05:50,13:15,7h 25m,2 stops,No info,7662.0,1,5,2019
2,Jet Airways,9/06/2019,Delhi,Cochin,DEL → LKO → BOM → COK,09:25,04:25,19h,2 stops,No info,13882.0,9,6,2019
3,IndiGo,12/05/2019,Kolkata,Banglore,CCU → NAG → BLR,18:05,23:30,5h 25m,1 stop,No info,6218.0,12,5,2019
4,IndiGo,01/03/2019,Banglore,New Delhi,BLR → NAG → DEL,16:50,21:35,4h 45m,1 stop,No info,13302.0,1,3,2019


In [73]:
#Converting Arrival_Time in hours and minutes

In [74]:
final_df['Arrival_Hour']=final_df['Arrival_Time'].apply(lambda x: x.split(':')[0])
final_df['Arrival_Minute']=final_df['Arrival_Time'].apply(lambda x: x.split(':')[1])

In [75]:
final_df.head()

,Airline,Date_of_Journey,Source,Destination,Route,Dep_Time,Arrival_Time,Duration,Total_Stops,Additional_Info,Price,Date,Month,Year,Arrival_Hour,Arrival_Minute
0,IndiGo,24/03/2019,Banglore,New Delhi,BLR → DEL,22:20,01:10,2h 50m,non-stop,No info,3897.0,24,3,2019,01,10
1,Air India,1/05/2019,Kolkata,Banglore,CCU → IXR → BBI → BLR,05:50,13:15,7h 25m,2 stops,No info,7662.0,1,5,2019,13,15
2,Jet Airways,9/06/2019,Delhi,Cochin,DEL → LKO → BOM → COK,09:25,04:25,19h,2 stops,No info,13882.0,9,6,2019,04,25
3,IndiGo,12/05/2019,Kolkata,Banglore,CCU → NAG → BLR,18:05,23:30,5h 25m,1 stop,No info,6218.0,12,5,2019,23,30
4,IndiGo,01/03/2019,Banglore,New Delhi,BLR → NAG → DEL,16:50,21:35,4h 45m,1 stop,No info,13302.0,1,3,2019,21,35


In [76]:
final_df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 13354 entries, 0 to 2670
Data columns (total 16 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   Airline          13354 non-null  object 
 1   Date_of_Journey  13354 non-null  object 
 2   Source           13354 non-null  object 
 3   Destination      13354 non-null  object 
 4   Route            13353 non-null  object 
 5   Dep_Time         13354 non-null  object 
 6   Arrival_Time     13354 non-null  object 
 7   Duration         13354 non-null  object 
 8   Total_Stops      13353 non-null  object 
 9   Additional_Info  13354 non-null  object 
 10  Price            10683 non-null  float64
 11  Date             13354 non-null  int64  
 12  Month            13354 non-null  int64  
 13  Year             13354 non-null  int64  
 14  Arrival_Hour     13354 non-null  object 
 15  Arrival_Minute   13354 non-null  object 
dtypes: float64(1), int64(3), object(12)
memory usage: 1.7+ MB


In [77]:
#Type cast Arrival_Hour & Arrival_Minute into integer Values

In [78]:
final_df['Arrival_Hour']=final_df['Arrival_Hour'].astype(int)
final_df['Arrival_Minute']=final_df['Arrival_Minute'].astype(int)
final_df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 13354 entries, 0 to 2670
Data columns (total 16 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   Airline          13354 non-null  object 
 1   Date_of_Journey  13354 non-null  object 
 2   Source           13354 non-null  object 
 3   Destination      13354 non-null  object 
 4   Route            13353 non-null  object 
 5   Dep_Time         13354 non-null  object 
 6   Arrival_Time     13354 non-null  object 
 7   Duration         13354 non-null  object 
 8   Total_Stops      13353 non-null  object 
 9   Additional_Info  13354 non-null  object 
 10  Price            10683 non-null  float64
 11  Date             13354 non-null  int64  
 12  Month            13354 non-null  int64  
 13  Year             13354 non-null  int64  
 14  Arrival_Hour     13354 non-null  int64  
 15  Arrival_Minute   13354 non-null  int64  
dtypes: float64(1), int64(5), object(10)
memory usage: 1.7+ MB


In [79]:
final_df.head()

,Airline,Date_of_Journey,Source,Destination,Route,Dep_Time,Arrival_Time,Duration,Total_Stops,Additional_Info,Price,Date,Month,Year,Arrival_Hour,Arrival_Minute
0,IndiGo,24/03/2019,Banglore,New Delhi,BLR → DEL,22:20,01:10,2h 50m,non-stop,No info,3897.0,24,3,2019,1,10
1,Air India,1/05/2019,Kolkata,Banglore,CCU → IXR → BBI → BLR,05:50,13:15,7h 25m,2 stops,No info,7662.0,1,5,2019,13,15
2,Jet Airways,9/06/2019,Delhi,Cochin,DEL → LKO → BOM → COK,09:25,04:25,19h,2 stops,No info,13882.0,9,6,2019,4,25
3,IndiGo,12/05/2019,Kolkata,Banglore,CCU → NAG → BLR,18:05,23:30,5h 25m,1 stop,No info,6218.0,12,5,2019,23,30
4,IndiGo,01/03/2019,Banglore,New Delhi,BLR → NAG → DEL,16:50,21:35,4h 45m,1 stop,No info,13302.0,1,3,2019,21,35


## Extract Departure Time Components

Departure time is separated into:

- Departure Hour
- Departure Minute

Splitting time into numerical components enables the model to learn travel patterns based on different times of the day.

In [84]:
final_df['Dept_Hour']=final_df['Dep_Time'].str.split(':').str[0]
final_df['Dept_Minute']=final_df['Dep_Time'].str.split(':').str[1]
final_df['Dept_Hour']=final_df['Dept_Hour'].astype(int)
final_df['Dept_Minute']=final_df['Dept_Minute'].astype(int)
final_df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 13354 entries, 0 to 2670
Data columns (total 18 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   Airline          13354 non-null  object 
 1   Date_of_Journey  13354 non-null  object 
 2   Source           13354 non-null  object 
 3   Destination      13354 non-null  object 
 4   Route            13353 non-null  object 
 5   Dep_Time         13354 non-null  object 
 6   Arrival_Time     13354 non-null  object 
 7   Duration         13354 non-null  object 
 8   Total_Stops      13353 non-null  object 
 9   Additional_Info  13354 non-null  object 
 10  Price            10683 non-null  float64
 11  Date             13354 non-null  int64  
 12  Month            13354 non-null  int64  
 13  Year             13354 non-null  int64  
 14  Arrival_Hour     13354 non-null  int64  
 15  Arrival_Minute   13354 non-null  int64  
 16  Dept_Hour        13354 non-null  int64  
 17  Dept_Minute      1

## Remove Redundant Features

After extracting useful temporal features, the original date and time columns become redundant.

Removing these columns reduces feature duplication and keeps the dataset cleaner for modeling.

In [137]:
df=final_df.drop(['Date_of_Journey','Arrival_Time','Dep_Time'],axis=1)
df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 13354 entries, 0 to 2670
Data columns (total 15 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   Airline          13354 non-null  object 
 1   Source           13354 non-null  object 
 2   Destination      13354 non-null  object 
 3   Route            13353 non-null  object 
 4   Duration         13354 non-null  object 
 5   Total_Stops      13353 non-null  object 
 6   Additional_Info  13354 non-null  object 
 7   Price            10683 non-null  float64
 8   Date             13354 non-null  int64  
 9   Month            13354 non-null  int64  
 10  Year             13354 non-null  int64  
 11  Arrival_Hour     13354 non-null  int64  
 12  Arrival_Minute   13354 non-null  int64  
 13  Dept_Hour        13354 non-null  int64  
 14  Dept_Minute      13354 non-null  int64  
dtypes: float64(1), int64(7), object(7)
memory usage: 2.1+ MB


In [138]:
df.head()

,Airline,Source,Destination,Route,Duration,Total_Stops,Additional_Info,Price,Date,Month,Year,Arrival_Hour,Arrival_Minute,Dept_Hour,Dept_Minute
0,IndiGo,Banglore,New Delhi,BLR → DEL,2h 50m,non-stop,No info,3897.0,24,3,2019,1,10,22,20
1,Air India,Kolkata,Banglore,CCU → IXR → BBI → BLR,7h 25m,2 stops,No info,7662.0,1,5,2019,13,15,5,50
2,Jet Airways,Delhi,Cochin,DEL → LKO → BOM → COK,19h,2 stops,No info,13882.0,9,6,2019,4,25,9,25
3,IndiGo,Kolkata,Banglore,CCU → NAG → BLR,5h 25m,1 stop,No info,6218.0,12,5,2019,23,30,18,5
4,IndiGo,Banglore,New Delhi,BLR → NAG → DEL,4h 45m,1 stop,No info,13302.0,1,3,2019,21,35,16,50


## Encode Number of Stops

The Total_Stops column is an ordinal categorical variable.

Instead of one-hot encoding, each category is mapped directly to an integer value:

- Non-stop → 0
- One Stop → 1
- Two Stops → 2
- Three Stops → 3
- Four Stops → 4

This preserves the natural ordering of the feature while making it suitable for machine learning algorithms.

In [139]:
#Handling categorical Variables
df['Total_Stops'].unique()

array(['non-stop', '2 stops', '1 stop', '3 stops', nan, '4 stops'],
      dtype=object)

In [140]:
df['Total_Stops'].isnull().sum()

np.int64(1)

In [141]:
df['Total_Stops'].value_counts()

Total_Stops
1 stop      7056
non-stop    4340
2 stops     1899
3 stops       56
4 stops        2
Name: count, dtype: int64

In [142]:
df['Total_Stops']=df['Total_Stops'].map({'non-stop':0,'1 stop':1,'2 stops':2, '3 stops':3, '4 stops':4})

In [143]:
df.head()

,Airline,Source,Destination,Route,Duration,Total_Stops,Additional_Info,Price,Date,Month,Year,Arrival_Hour,Arrival_Minute,Dept_Hour,Dept_Minute
0,IndiGo,Banglore,New Delhi,BLR → DEL,2h 50m,0.0,No info,3897.0,24,3,2019,1,10,22,20
1,Air India,Kolkata,Banglore,CCU → IXR → BBI → BLR,7h 25m,2.0,No info,7662.0,1,5,2019,13,15,5,50
2,Jet Airways,Delhi,Cochin,DEL → LKO → BOM → COK,19h,2.0,No info,13882.0,9,6,2019,4,25,9,25
3,IndiGo,Kolkata,Banglore,CCU → NAG → BLR,5h 25m,1.0,No info,6218.0,12,5,2019,23,30,18,5
4,IndiGo,Banglore,New Delhi,BLR → NAG → DEL,4h 45m,1.0,No info,13302.0,1,3,2019,21,35,16,50


In [144]:
df['Total_Stops']=df['Total_Stops'].fillna(1)

In [145]:
df['Total_Stops'].isnull().sum()

np.int64(0)

In [146]:
#Dropping route column as it will not be needed as part of analysis

In [147]:
df.drop('Route', axis=1,inplace=True)
df.head()

,Airline,Source,Destination,Duration,Total_Stops,Additional_Info,Price,Date,Month,Year,Arrival_Hour,Arrival_Minute,Dept_Hour,Dept_Minute
0,IndiGo,Banglore,New Delhi,2h 50m,0.0,No info,3897.0,24,3,2019,1,10,22,20
1,Air India,Kolkata,Banglore,7h 25m,2.0,No info,7662.0,1,5,2019,13,15,5,50
2,Jet Airways,Delhi,Cochin,19h,2.0,No info,13882.0,9,6,2019,4,25,9,25
3,IndiGo,Kolkata,Banglore,5h 25m,1.0,No info,6218.0,12,5,2019,23,30,18,5
4,IndiGo,Banglore,New Delhi,4h 45m,1.0,No info,13302.0,1,3,2019,21,35,16,50


## Convert Flight Duration into Numerical Features

Flight duration is originally stored as text (e.g., "2h 50m").

To make this information usable by machine learning models:

- Hours are extracted
- Minutes are extracted
- Total flight duration is calculated in minutes

Representing duration as a single numerical feature improves model interpretability and performance.

In [149]:
df['Duration_Hours']=df['Duration'].str.split(' ').str[0].str.split('h').str[0]

In [150]:
df.head(2)

,Airline,Source,Destination,Duration,Total_Stops,Additional_Info,Price,Date,Month,Year,Arrival_Hour,Arrival_Minute,Dept_Hour,Dept_Minute,Duration_Hours
0,IndiGo,Banglore,New Delhi,2h 50m,0.0,No info,3897.0,24,3,2019,1,10,22,20,2
1,Air India,Kolkata,Banglore,7h 25m,2.0,No info,7662.0,1,5,2019,13,15,5,50,7


In [151]:
df[df['Duration_Hours']=='5m']

,Airline,Source,Destination,Duration,Total_Stops,Additional_Info,Price,Date,Month,Year,Arrival_Hour,Arrival_Minute,Dept_Hour,Dept_Minute,Duration_Hours
6474,Air India,Mumbai,Hyderabad,5m,2.0,No info,17327.0,6,3,2019,16,55,16,50,5m
2660,Air India,Mumbai,Hyderabad,5m,2.0,No info,NaN,12,3,2019,16,55,16,50,5m


In [152]:
df.drop(6474, axis=0, inplace=True)
df.drop(2660, axis=0, inplace=True)

In [153]:
df[df['Duration_Hours']=='5m']

,Airline,Source,Destination,Duration,Total_Stops,Additional_Info,Price,Date,Month,Year,Arrival_Hour,Arrival_Minute,Dept_Hour,Dept_Minute,Duration_Hours


In [154]:
#Converting hours to int
df['Duration_Hours']=df['Duration_Hours'].astype(int)
df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 13351 entries, 0 to 2670
Data columns (total 15 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   Airline          13351 non-null  object 
 1   Source           13351 non-null  object 
 2   Destination      13351 non-null  object 
 3   Duration         13351 non-null  object 
 4   Total_Stops      13351 non-null  float64
 5   Additional_Info  13351 non-null  object 
 6   Price            10681 non-null  float64
 7   Date             13351 non-null  int64  
 8   Month            13351 non-null  int64  
 9   Year             13351 non-null  int64  
 10  Arrival_Hour     13351 non-null  int64  
 11  Arrival_Minute   13351 non-null  int64  
 12  Dept_Hour        13351 non-null  int64  
 13  Dept_Minute      13351 non-null  int64  
 14  Duration_Hours   13351 non-null  int64  
dtypes: float64(2), int64(8), object(5)
memory usage: 1.6+ MB


In [155]:
#Obtaining minutes from duration by converting null values to zero and converting to int

In [156]:
df['Duration_Minutes']=df['Duration'].str.split(' ').str[1].str.split('m').str[0].replace(np.nan,0).astype(int)
df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 13351 entries, 0 to 2670
Data columns (total 16 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   Airline           13351 non-null  object 
 1   Source            13351 non-null  object 
 2   Destination       13351 non-null  object 
 3   Duration          13351 non-null  object 
 4   Total_Stops       13351 non-null  float64
 5   Additional_Info   13351 non-null  object 
 6   Price             10681 non-null  float64
 7   Date              13351 non-null  int64  
 8   Month             13351 non-null  int64  
 9   Year              13351 non-null  int64  
 10  Arrival_Hour      13351 non-null  int64  
 11  Arrival_Minute    13351 non-null  int64  
 12  Dept_Hour         13351 non-null  int64  
 13  Dept_Minute       13351 non-null  int64  
 14  Duration_Hours    13351 non-null  int64  
 15  Duration_Minutes  13351 non-null  int64  
dtypes: float64(2), int64(9), object(5)
memory usag

In [157]:
df.head(2)

,Airline,Source,Destination,Duration,Total_Stops,Additional_Info,Price,Date,Month,Year,Arrival_Hour,Arrival_Minute,Dept_Hour,Dept_Minute,Duration_Hours,Duration_Minutes
0,IndiGo,Banglore,New Delhi,2h 50m,0.0,No info,3897.0,24,3,2019,1,10,22,20,2,50
1,Air India,Kolkata,Banglore,7h 25m,2.0,No info,7662.0,1,5,2019,13,15,5,50,7,25


In [158]:
df['Total_Duration']=(df['Duration_Hours']*60)+df['Duration_Minutes']

In [159]:
df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 13351 entries, 0 to 2670
Data columns (total 17 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   Airline           13351 non-null  object 
 1   Source            13351 non-null  object 
 2   Destination       13351 non-null  object 
 3   Duration          13351 non-null  object 
 4   Total_Stops       13351 non-null  float64
 5   Additional_Info   13351 non-null  object 
 6   Price             10681 non-null  float64
 7   Date              13351 non-null  int64  
 8   Month             13351 non-null  int64  
 9   Year              13351 non-null  int64  
 10  Arrival_Hour      13351 non-null  int64  
 11  Arrival_Minute    13351 non-null  int64  
 12  Dept_Hour         13351 non-null  int64  
 13  Dept_Minute       13351 non-null  int64  
 14  Duration_Hours    13351 non-null  int64  
 15  Duration_Minutes  13351 non-null  int64  
 16  Total_Duration    13351 non-null  int64  
dtyp

In [160]:
#Dropping Duration, Duration_Hours, Duration_Minutes

In [161]:
df.drop(['Duration','Duration_Hours','Duration_Minutes'], axis=1, inplace=True)

In [162]:
df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 13351 entries, 0 to 2670
Data columns (total 14 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   Airline          13351 non-null  object 
 1   Source           13351 non-null  object 
 2   Destination      13351 non-null  object 
 3   Total_Stops      13351 non-null  float64
 4   Additional_Info  13351 non-null  object 
 5   Price            10681 non-null  float64
 6   Date             13351 non-null  int64  
 7   Month            13351 non-null  int64  
 8   Year             13351 non-null  int64  
 9   Arrival_Hour     13351 non-null  int64  
 10  Arrival_Minute   13351 non-null  int64  
 11  Dept_Hour        13351 non-null  int64  
 12  Dept_Minute      13351 non-null  int64  
 13  Total_Duration   13351 non-null  int64  
dtypes: float64(2), int64(8), object(4)
memory usage: 1.5+ MB


## Encode Categorical Variables

Machine learning algorithms require numerical input features.

Nominal categorical variables such as:

- Airline
- Source
- Destination
- Additional Information

are transformed using One-Hot Encoding.

One-Hot Encoding creates binary indicator columns for each category while preventing the model from assuming any ordinal relationship between categories.

In [173]:
from sklearn.preprocessing import OneHotEncoder

In [174]:
cat_cols=['Airline', 'Source', 'Destination', 'Additional_Info']

In [175]:
ohe = OneHotEncoder(sparse_output=False, handle_unknown='ignore')

In [176]:
encoded_array = ohe.fit_transform(df[cat_cols])

In [177]:
encoded_df = pd.DataFrame(
    encoded_array,
    columns=ohe.get_feature_names_out(cat_cols),
    index=df.index
)

In [178]:
df = pd.concat([df.drop(columns=cat_cols), encoded_df], axis=1)

In [179]:
df.head()

,Total_Stops,Price,Date,Month,Year,Arrival_Hour,Arrival_Minute,Dept_Hour,Dept_Minute,Total_Duration,...,Additional_Info_1 Long layover,Additional_Info_1 Short layover,Additional_Info_2 Long layover,Additional_Info_Business class,Additional_Info_Change airports,Additional_Info_In-flight meal not included,Additional_Info_No Info,Additional_Info_No check-in baggage included,Additional_Info_No info,Additional_Info_Red-eye flight
0,0.0,3897.0,24,3,2019,1,10,22,20,170,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0
1,2.0,7662.0,1,5,2019,13,15,5,50,445,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0
2,2.0,13882.0,9,6,2019,4,25,9,25,1140,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0
3,1.0,6218.0,12,5,2019,23,30,18,5,325,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0
4,1.0,13302.0,1,3,2019,21,35,16,50,285,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0


# Data Preprocessing Summary

The preprocessing pipeline completed the following steps:

- Loaded training and test datasets
- Combined both datasets for consistent preprocessing
- Performed exploratory data inspection
- Extracted date-related features
- Extracted departure and arrival time components
- Converted flight duration into numerical values
- Encoded the number of stops
- Removed redundant features
- Applied One-Hot Encoding to categorical variables

The resulting dataset is fully numerical and ready for machine learning model development and flight price prediction.